# 🏎️ Formula 1 Race Intelligence - Exploratory Data Analysis & ML Models
Welcome to the F1 Race Intelligence exploratory notebook. This notebook demonstrates how to load FastF1 timing and telemetry data, perform distance-based synchronization, model tire degradation with fuel burn compensation, and train time-aware lap-time regression models.

In [ ]:
# Install dependencies in Google Colab
!pip install fastf1 plotly scikit-learn pandas numpy scipy matplotlib

In [ ]:
import os
import fastf1
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Setup cache directory for FastF1
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")
print("FastF1 cache enabled.")

## 1. Load Session Data (2024 Bahrain Grand Prix)

In [ ]:
# Load Race session
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load(laps=True, telemetry=True, weather=True)

laps = session.laps
laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
print(f"Loaded {len(laps)} total laps from {session.event.EventName}")

## 2. Telemetry Synchronization & Delta Analysis (VER vs LEC)

In [ ]:
# Extract fastest laps for Verstappen and Leclerc
lap_ver = laps.pick_driver('VER').pick_fastest()
lap_lec = laps.pick_driver('LEC').pick_fastest()

tel_ver = lap_ver.get_telemetry()
tel_lec = lap_lec.get_telemetry()

# Synchronize onto uniform distance grid
max_dist = min(tel_ver['Distance'].max(), tel_lec['Distance'].max())
grid_dist = np.linspace(0, max_dist, 1000)

spd_ver = np.interp(grid_dist, tel_ver['Distance'], tel_ver['Speed'])
spd_lec = np.interp(grid_dist, tel_lec['Distance'], tel_lec['Speed'])
delta_spd = spd_ver - spd_lec

fig = go.Figure()
fig.add_trace(go.Scatter(x=grid_dist, y=spd_ver, name='VER Speed (km/h)', line=dict(color='#00D2BE')))
fig.add_trace(go.Scatter(x=grid_dist, y=spd_lec, name='LEC Speed (km/h)', line=dict(color='#E10600')))
fig.update_layout(title='Distance-Synchronized Speed Comparison', xaxis_title='Distance (m)', yaxis_title='Speed (km/h)', template='plotly_dark')
fig.show()

## 3. Tire Degradation with Fuel Burn Correction

In [ ]:
# Filter clean laps
clean = laps.dropna(subset=['LapTimeSeconds', 'Compound'])
clean = clean[clean['TrackStatus'] == '1']

# Apply fuel correction (~0.050s faster per lap)
clean['FuelCorrectedPace'] = clean['LapTimeSeconds'] + (clean['LapNumber'] * 0.050)

# Fit regression for each compound
for comp in clean['Compound'].unique():
    sub = clean[clean['Compound'] == comp]
    if len(sub) > 10:
        slope, intercept, r_val, p_val, std_err = stats.linregress(sub['TyreLife'], sub['FuelCorrectedPace'])
        print(f"{comp} Tire Wear: +{slope:.4f} s/lap (R²: {r_val**2:.3f}, p: {p_val:.4f})")

## 4. Time-Aware Machine Learning Lap Time Prediction

In [ ]:
# Chronological Train/Test Split (Laps 1-38 Train, Laps 39-57 Test)
clean = clean.sort_values(['Driver', 'LapNumber'])
clean['Lag1'] = clean.groupby('Driver')['LapTimeSeconds'].shift(1).fillna(clean['LapTimeSeconds'])
clean['FuelKg'] = np.maximum(5.0, 105.0 - clean['LapNumber'] * 1.85)

features = ['LapNumber', 'TyreLife', 'FuelKg', 'Lag1']
train = clean[clean['LapNumber'] <= 38]
test = clean[clean['LapNumber'] > 38]

rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf.fit(train[features], train['LapTimeSeconds'])

preds = rf.predict(test[features])
mae = mean_absolute_error(test['LapTimeSeconds'], preds)
r2 = r2_score(test['LapTimeSeconds'], preds)

print(f"Holdout Test Evaluation:")
print(f"MAE: {mae:.3f} seconds")
print(f"R² Score: {r2:.3f}")